[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Open-Athena/MarinFold/blob/main/notebooks/inference_example1.ipynb) [![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/Open-Athena/MarinFold/blob/main/notebooks/inference_example1.ipynb)

# MarinFold Contact-Map Demo

This notebook clones or reuses the repo, installs the packaged `marinfold` library with the right optional extra for the current hardware, runs our current best **contacts-v1 1.5B** model on an mmCIF from RCSB, and plots the ground-truth vs predicted **residue–residue contact map** inline.

The model predicts contacts from sequence alone — no MSA, no template, no structure. On a free GPU (Colab T4 or Kaggle T4) it runs on the `transformers` backend; see the [project README](https://github.com/Open-Athena/MarinFold#try-it-out).

## Before You Run

Runtime selection is automatic — matched to the hardware and, for NVIDIA, its compute capability:

- **Modern NVIDIA GPU** (Ampere or newer, compute capability ≥ 8.0 — A100, L4, H100, RTX 30/40/50…) → MarinFold `vllm` (fastest).
- **Older NVIDIA GPU** (Turing/Pascal, compute capability < 8.0 — **Colab's free T4**, Kaggle T4/P100) → MarinFold `transformers`, GPU-accelerated. Recent vLLM wheels no longer run on these cards, so we drive the GPU with torch directly — correct, and still on the GPU.
- Apple Silicon → MarinFold `mlx` (pairwise only; rollout needs vllm/transformers).
- CPU / other → MarinFold `transformers`.
- **TPU → runs on the TPU VM's host CPU** via `transformers`. MarinFold has no in-notebook TPU-*chip* backend (vLLM-TPU needs a from-source build; PyTorch/XLA on Colab is unsupported), so a TPU runtime uses its host CPU here — correct, but **not** TPU-accelerated.

So the **default Colab free T4 runs on the `transformers` GPU path** and the pairwise demo below finishes in a few seconds. `vllm` is reserved for Ampere+ GPUs (e.g. Colab Pro's L4/A100), where the install cell fetches a **CUDA-12 vLLM wheel** that matches Colab's driver — the default PyPI wheel is CUDA-13 and fails with `ImportError: libcudart.so.13`. Override the choice with `BACKEND_MODE` in the config; a forced-but-unusable backend falls back to `transformers` automatically.

**No NVIDIA GPU on Colab right now?** Try again later for a **T4**, or use **[Kaggle](https://www.kaggle.com/code)** (free T4/P100, often better availability — this notebook runs there unchanged). For real TPU-*chip* speed, a GCP TPU VM with MarinFold's vLLM-TPU recipe is the path (outside Colab).

After the install cell, Colab may need a runtime restart before the new packages are visible; if `import marinfold` fails, do **Runtime -> Restart session** and rerun from the import cell.

Two inference methods (`METHOD` in the config below):

- **`pairwise`** (default, seconds on GPU) — scores `P(contact)` for every residue pair.
- **`rollout`** (exp82's best, much slower) — votes over sampled contact-section completions with a pairwise tie-break. Needs `vllm` or `transformers` (not `mlx`); set `N_ROLLOUTS` to trade speed for quality.

In [ ]:
# @title Configuration
MODEL_NICKNAME = "contacts-v1-exp75-1.5B"  # @param ["contacts-v1-exp75-1.5B"]
PDB_ID = "1QYS"  # @param {type:"string"}
CHAIN = ""  # @param {type:"string"}
METHOD = "pairwise"  # @param ["pairwise", "rollout"]
N_ROLLOUTS = 100  # @param {type:"integer"}
ENSEMBLE_K = 1  # @param {type:"integer"}
BACKEND_MODE = "auto"  # @param ["auto", "vllm", "mlx", "transformers"]
DTYPE = "bfloat16"  # @param ["bfloat16", "float16", "float32"]
BATCH_SIZE = 32  # @param {type:"integer"}
GPU_MEMORY_UTILIZATION = 0.8  # @param {type:"number"}

# CHAIN picks which author chain to run — contacts-v1 is a single-chain model.
# Leave blank for a single-chain entry (or to auto-pick the first of several);
# for a complex like PDB_ID="3GSS" set CHAIN="A" or "B".

print(
    {
        "model": MODEL_NICKNAME,
        "pdb_id": PDB_ID.upper(),
        "chain": CHAIN.strip() or "(auto)",
        "method": METHOD,
        "n_rollouts": N_ROLLOUTS,
        "ensemble_k": ENSEMBLE_K,
        "backend_mode": BACKEND_MODE,
        "dtype": DTYPE,
        "batch_size": BATCH_SIZE,
        "gpu_memory_utilization": GPU_MEMORY_UTILIZATION,
    }
)

MarinFold's contacts-v1 is a **single-chain** model. For a multi-chain entry — e.g. `3GSS` has chains A and B — set **`CHAIN`** in the config to pick one (that chain is isolated into its own structure before scoring); leave `CHAIN` blank for a single-chain entry, or to auto-select the first chain of several.

`evaluate` compares the model's predicted contacts against **pyconfind** side-chain ground-truth contacts on the structure (degree ≥ 0.001, sequence-separation ≥ 6 — the contacts-v1 definition). It reports ranking AUC and precision@{L, L/2, L/5}, and the plot below shows the ground-truth contact map next to the model's predicted contact score. The ground-truth step needs the `contacts-v1` extra (pyconfind), which the install cell adds automatically.

In [ ]:
import os
import platform
import shutil
import subprocess
from pathlib import Path


def _find_repo_root(start: Path):
    for candidate in [start, *start.parents]:
        if (candidate / "README.md").exists() and (candidate / "marinfold" / "pyproject.toml").exists():
            return candidate
    return None


def _has_nvidia_gpu() -> bool:
    if shutil.which("nvidia-smi") is None:
        return False
    result = subprocess.run(
        ["nvidia-smi", "-L"],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        check=False,
    )
    return result.returncode == 0 and bool(result.stdout.strip())


def _nvidia_compute_capability():
    """Highest NVIDIA compute capability present (e.g. 7.5 for a T4), or None.

    vLLM's current wheels only run on Ampere or newer (>= 8.0). Turing/Pascal
    cards -- Colab's free T4, Kaggle's T4/P100 -- must use the transformers
    path instead, so this gates the backend choice below.
    """
    if shutil.which("nvidia-smi") is None:
        return None
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        check=False,
    )
    if result.returncode != 0:
        return None
    caps = []
    for line in result.stdout.splitlines():
        try:
            caps.append(float(line.strip()))
        except ValueError:
            continue
    return max(caps) if caps else None


def _has_tpu() -> bool:
    return any(
        os.environ.get(key)
        for key in ("COLAB_TPU_ADDR", "TPU_NAME", "TPU_WORKER_ID")
    )


def _is_apple_silicon() -> bool:
    return platform.system() == "Darwin" and platform.machine() == "arm64"


def _auto_backend_choice():
    if _has_nvidia_gpu():
        cap = _nvidia_compute_capability()
        # vLLM's current wheels need Ampere+ (compute capability >= 8.0).
        # Turing/Pascal (T4, P100 -- the Colab/Kaggle free tiers) fail on vLLM,
        # so run on the GPU via transformers there. Unknown capability -> the
        # safe transformers path.
        if cap is not None and cap >= 8.0:
            return {
                "backend": "vllm",
                "extra": "vllm",
                "runtime": f"nvidia-gpu-sm{cap:g}",
                "note": f"Using vLLM: NVIDIA GPU compute capability {cap:g} (>= 8.0).",
            }
        cap_str = f"{cap:g}" if cap is not None else "unknown"
        return {
            "backend": "transformers",
            "extra": "transformers",
            "runtime": f"nvidia-gpu-sm{cap_str}",
            "note": (
                f"NVIDIA GPU compute capability {cap_str} (< 8.0, e.g. a Colab/"
                "Kaggle T4) -> running on the GPU via transformers. Recent vLLM "
                "wheels need Ampere+; force BACKEND_MODE='vllm' only on such a GPU."
            ),
        }
    if _has_tpu():
        # No in-notebook TPU-*chip* backend (vLLM-TPU needs a source build;
        # PyTorch/XLA on Colab is unsupported). A Colab TPU VM has a capable
        # HOST CPU, so run there via transformers -- correct, just not TPU-
        # accelerated. For speed use a GPU runtime, Kaggle, or a GCP TPU VM.
        return {
            "backend": "transformers",
            "extra": "transformers",
            "runtime": "tpu-host-cpu",
            "note": (
                "TPU runtime detected -> running on the TPU VM's HOST CPU via "
                "transformers (not the TPU chips). For acceleration use a GPU "
                "runtime, Kaggle's free GPUs, or a GCP TPU VM with vLLM-TPU."
            ),
        }
    if _is_apple_silicon():
        return {
            "backend": "mlx",
            "extra": "mlx",
            "runtime": "apple-silicon",
            "note": "Using MLX because Apple Silicon is available.",
        }
    return {
        "backend": "transformers",
        "extra": "transformers",
        "runtime": "cpu-or-other",
        "note": "Falling back to transformers because no supported accelerator was detected.",
    }


BASE_WORKDIR = Path("/content") if Path("/content").exists() and os.access("/content", os.W_OK) else Path.cwd()
REPO_DIR = _find_repo_root(Path.cwd()) or (BASE_WORKDIR / "MarinFold")
if not REPO_DIR.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/Open-Athena/MarinFold.git",
            str(REPO_DIR),
        ],
        check=True,
    )

AUTO_BACKEND = _auto_backend_choice()
if BACKEND_MODE == "auto":
    BACKEND_NAME = AUTO_BACKEND["backend"]
    EXTRA_NAME = AUTO_BACKEND["extra"]
else:
    BACKEND_NAME = BACKEND_MODE
    EXTRA_NAME = BACKEND_MODE

print(f"repo checkout: {REPO_DIR}")
print({
    "backend": BACKEND_NAME,
    "extra": EXTRA_NAME,
    "runtime": AUTO_BACKEND["runtime"],
    "note": AUTO_BACKEND["note"],
})

In [ ]:
import importlib.util
import shutil
import subprocess
import sys

PACKAGE_DIR = REPO_DIR / "marinfold"

# Base installer: prefer pip, fall back to uv.
if importlib.util.find_spec("pip") is not None:
    _PIP = [sys.executable, "-m", "pip", "install", "-q"]
elif shutil.which("uv") is not None:
    _PIP = ["uv", "pip", "install", "--python", sys.executable, "-q"]
else:
    raise RuntimeError("Neither pip nor uv is available to install notebook dependencies.")

if EXTRA_NAME == "vllm":
    # Colab / most cloud drivers are CUDA 12.x, but vLLM's default PyPI wheel is
    # now built for CUDA 13 -> `ImportError: libcudart.so.13`. Pull a pinned
    # CUDA-12.9 vLLM wheel from vLLM's wheel index (matches the driver; cu129
    # wheels also run on newer CUDA-13 drivers) instead of the bare `[vllm]`
    # extra. The `+cu129` local version can only be satisfied by that index, and
    # pip resolves the index's content-addressed wheel links for us. Only reached
    # on Ampere+ GPUs -- older cards use transformers above.
    #
    # Also install the `transformers` extra as a runtime fallback backend. The
    # marinfold editable install and the vLLM pin go in ONE command so pip/uv
    # resolves transformers once against both constraints (marinfold's `<5` and
    # vLLM's), landing on a version that satisfies each.
    VLLM_VERSION = "0.20.2"
    EXTRAS = "transformers,contacts-v1"
    install_args = [
        "-e", f".[{EXTRAS}]",
        f"vllm=={VLLM_VERSION}+cu129",
        "--extra-index-url", f"https://wheels.vllm.ai/{VLLM_VERSION}/cu129/",
        "--extra-index-url", "https://download.pytorch.org/whl/cu129",
    ]
else:
    # Install the chosen backend extra plus `contacts-v1` (pyconfind), which the
    # evaluate path needs to read ground-truth contacts from the structure.
    EXTRAS = f"{EXTRA_NAME},contacts-v1"
    install_args = ["-e", f".[{EXTRAS}]"]

subprocess.run(_PIP + install_args, cwd=PACKAGE_DIR, check=True)

# Make the freshly-installed package importable in THIS kernel without a
# restart: put the source dir on sys.path (its deps were just installed above)
# and drop cached import finders. Colab sometimes doesn't see an editable
# install until a restart otherwise.
if str(PACKAGE_DIR) not in sys.path:
    sys.path.insert(0, str(PACKAGE_DIR))
importlib.invalidate_caches()

_label = f"{EXTRAS} + vllm=={VLLM_VERSION}(cu129)" if EXTRA_NAME == "vllm" else EXTRAS
print(f"installed marinfold extras: {_label}")
print(
    "If the next cell still raises ModuleNotFoundError, do "
    "Runtime -> Restart session and rerun from the import cell (skip clone/install)."
)

In [ ]:
import importlib
import json
import os
import platform
import time
from pathlib import Path
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import numpy as np

from marinfold import write_eval
from marinfold.document_structures.contacts_v1 import (
    InferenceConfig,
    evaluate,
    plot_evaluate_pdf,
)
from marinfold.registry import list_model_entries, resolve_model_entry

os.environ.setdefault("VLLM_LOGGING_LEVEL", "WARNING")

runtime_summary = {
    "backend": BACKEND_NAME,
    "runtime": AUTO_BACKEND["runtime"],
    "platform": platform.platform(),
    "machine": platform.machine(),
}

torch_spec = importlib.util.find_spec("torch")
if torch_spec is not None:
    import torch

    runtime_summary["torch_version"] = torch.__version__
    runtime_summary["cuda_available"] = torch.cuda.is_available()
    if torch.cuda.is_available():
        props = torch.cuda.get_device_properties(0)
        runtime_summary["gpu_name"] = props.name
        runtime_summary["gpu_memory_gib"] = round(props.total_memory / (1024 ** 3), 1)

print(runtime_summary)

In [ ]:
entries = {entry.nickname: entry for entry in list_model_entries()}
print("Available model nicknames:", sorted(entries))

entry = resolve_model_entry(MODEL_NICKNAME)
print(f"Using model: {entry.nickname}")
print(f"HF URL: {entry.url}")
if entry.wandb_url:
    print(f"W&B: {entry.wandb_url}")


In [ ]:
import gemmi

RUN_DIR = BASE_WORKDIR / "marinfold_notebook_runs"
RUN_DIR.mkdir(parents=True, exist_ok=True)
raw_cif = RUN_DIR / f"{PDB_ID.upper()}.cif"
cif_url = f"https://files.rcsb.org/download/{PDB_ID.upper()}.cif"

if not raw_cif.exists():
    urlretrieve(cif_url, raw_cif)


def _protein_chains(structure):
    """Author chain ids in model 1 that hold >= 2 amino-acid residues."""
    names = []
    for chain in structure[0]:
        n_aa = 0
        for res in chain:
            info = gemmi.find_tabulated_residue(res.name)
            if info is not None and info.is_amino_acid():
                n_aa += 1
        if n_aa >= 2:
            names.append(chain.name)
    return names


# contacts-v1 is single-chain. Choose the author chain to score: CHAIN if given
# (e.g. "A"/"B" for a complex like 3GSS), else the entry's sole chain, else the
# first of several (with a note). A multi-chain entry -- or an explicit CHAIN --
# is isolated into its own single-chain mmCIF; a genuinely single-chain entry
# with no CHAIN set is used as downloaded.
structure = gemmi.read_structure(str(raw_cif))
chains = _protein_chains(structure)
if not chains:
    raise ValueError(f"{PDB_ID.upper()}: no protein chains found in the structure.")

requested = CHAIN.strip()
if requested:
    if requested not in chains:
        raise ValueError(
            f"{PDB_ID.upper()}: chain {requested!r} not found. "
            f"Protein chains in this entry: {chains}."
        )
    selected = requested
else:
    selected = chains[0]
    if len(chains) > 1:
        print(
            f"{PDB_ID.upper()} has {len(chains)} protein chains {chains}; using "
            f"{selected!r}. Set CHAIN in the config cell to pick another."
        )

if len(chains) > 1 or requested:
    entry_id = f"{PDB_ID.upper()}_{selected}"
    INPUT_PATH = RUN_DIR / f"{entry_id}.cif"
    single = gemmi.Selection(f"//{selected}").copy_structure_selection(structure)
    single.setup_entities()
    single.name = entry_id
    single.make_mmcif_document().write_file(str(INPUT_PATH))
else:
    INPUT_PATH = raw_cif

print(f"input structure: {INPUT_PATH}")
print(f"chain: {selected}   (protein chains in entry: {chains})")
print(f"source url: {cif_url}")


In [ ]:
from dataclasses import replace

cfg = InferenceConfig(
    model=MODEL_NICKNAME,
    input_path=INPUT_PATH,
    backend=BACKEND_NAME,
    method=METHOD,
    n_rollouts=N_ROLLOUTS,
    ensemble_k=ENSEMBLE_K,
    batch_size=BATCH_SIZE,
    dtype=DTYPE,
    gpu_memory_utilization=GPU_MEMORY_UTILIZATION,
)


def _evaluate_with_fallback(config):
    """Run evaluate(); if a non-transformers backend can't load or run (e.g. a
    vLLM CUDA / GPU-architecture mismatch), retry once on the transformers
    backend so the demo still produces a result instead of a raw traceback."""
    try:
        return config, evaluate(config)
    except Exception as exc:  # noqa: BLE001 - demo: degrade instead of crashing
        if config.backend == "transformers":
            raise
        print(
            f"[warn] backend {config.backend!r} failed to run "
            f"({type(exc).__name__}: {exc})."
        )
        print("[warn] retrying on the transformers backend (GPU if available, else CPU).")
        fallback = replace(config, backend="transformers")
        return fallback, evaluate(fallback)


t0 = time.time()
cfg, result = _evaluate_with_fallback(cfg)
elapsed = time.time() - t0
BACKEND_NAME = cfg.backend  # reflect any fallback in the cells below

# Headline long-range (seq-sep >= 24) contact-prediction metrics.
long_metrics = {
    k.rsplit("_", 1)[0]: v for k, v in result.metrics.items() if k.endswith("_long")
}
print(f"backend: {cfg.backend}")
print(json.dumps(long_metrics, indent=2))
print(f"elapsed_seconds: {elapsed:.1f}")

In [ ]:
def reconstruct_matrix(rows, value_key, n_residues):
    out = np.full((n_residues, n_residues), np.nan, dtype=np.float32)
    for row in rows:
        i = int(row["i"]) - 1
        j = int(row["j"]) - 1
        value = float(row[value_key])
        out[i, j] = value
        out[j, i] = value
    return out


rows = result.per_example
if not rows:
    raise ValueError("No per-example rows were produced.")

entry_id = rows[0]["entry_id"]
n_residues = int(result.extras["per_structure_n_residues"][entry_id])
method = result.extras.get("method", METHOD)
score_label = "rollout vote score" if method == "rollout" else "P(contact)"
auc_long = float(result.metrics.get("auc_long", float("nan")))

gt = reconstruct_matrix(rows, "gt", n_residues)
pred = reconstruct_matrix(rows, "score", n_residues)
n_contacts = int(np.nansum(gt) // 2)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))
axes[0].imshow(gt, cmap="Greys", origin="lower", interpolation="none")
axes[0].set_title(f"ground-truth contacts\n(L={n_residues}, {n_contacts} contacts, pyconfind)")
vmax = float(np.nanpercentile(pred, 99.5)) or float(np.nanmax(pred)) or 1.0
im = axes[1].imshow(pred, cmap="magma", origin="lower", vmin=0, vmax=vmax, interpolation="none")
axes[1].set_title(f"MarinFold-cv1 · {score_label}\n({method} readout, from sequence)")
fig.colorbar(im, ax=axes[1], fraction=0.046, label=score_label)

for ax in axes:
    ax.set_xlabel("residue j")
    ax.set_ylabel("residue i")

fig.suptitle(
    f"{entry_id} | model={MODEL_NICKNAME} | method={method} | long-range AUC = {auc_long:.3f}",
    fontsize=13,
)
fig.tight_layout()
plt.show()

In [ ]:
OUTPUT_DIR = RUN_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

safe_model = MODEL_NICKNAME.replace(".", "_")
base_name = f"{entry_id}_{safe_model}_{method}"

metrics_path = OUTPUT_DIR / f"{base_name}_metrics.json"
plots_pdf_path = OUTPUT_DIR / f"{base_name}_contacts.pdf"
heatmap_png_path = OUTPUT_DIR / f"{base_name}_heatmap.png"

write_eval(metrics_path, result, structure_name="contacts-v1")
plot_evaluate_pdf(plots_pdf_path, result)
fig.savefig(heatmap_png_path, dpi=150, bbox_inches="tight")

print(f"saved metrics: {metrics_path}")
print(f"saved pdf: {plots_pdf_path}")
print(f"saved png: {heatmap_png_path}")